In [1]:
from pathlib import Path
import pyreadr
import pandas as pd
import numpy as np
import json

OUT_ROOT = Path("test_2")
DATA_DIR = OUT_ROOT / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from pathlib import Path
import pyreadr
import pandas as pd
import numpy as np
import json

OUT_ROOT = Path("test_2")
DATA_DIR = OUT_ROOT / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
result = pyreadr.read_r("data/atsp_spring.rds")
df = list(result.values())[0]

df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date

df["bird_id"] = (
    df["BandNumber"].astype(str) + "_" +
    df["motusTagID"].astype(int).astype(str)
)

df["minute_of_day"] = df["hour"] * 60 + df["minute"]

df["antenna_id"] = (
    df["recvDeployName"].astype(str) +
    "_P" + df["port"].astype(int).astype(str)
)

df["antenna_id"] = df["antenna_id"].str.replace(" ", "_")

df["recvDeployLat"] = pd.to_numeric(df["recvDeployLat"], errors="coerce")
df["recvDeployLon"] = pd.to_numeric(df["recvDeployLon"], errors="coerce")

df = df.dropna(subset=[
    "bird_id","date","minute_of_day",
    "antenna_id","sig",
    "recvDeployLat","recvDeployLon"
]).copy()

In [4]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [5]:
def build_day(df, bird_id, day, top_k=3):
    d = df[(df["bird_id"] == bird_id) & (df["date"] == day)].copy()
    if d.empty:
        return None

    # ---- BASE tower (first detection)
    first = d.sort_values("ts").iloc[0]
    base_lat = first["recvDeployLat"]
    base_lon = first["recvDeployLon"]

    # ---- MERGED
    d = d.sort_values("ts")
    d["delta_sig"] = d.groupby("antenna_id")["sig"].diff()
    d["activity"] = np.sqrt(np.abs(d["delta_sig"]))

    merged = (
        d.groupby("minute_of_day")
         .agg(activity=("activity", "median"))
    )

    merged = (
        pd.DataFrame({"minute_of_day": np.arange(1440)})
        .merge(merged, on="minute_of_day", how="left")
        .fillna(0)
    )

    # ---- RAW (top 3 antennas)
    top_ant = (
        d.groupby("antenna_id")["sig"]
        .size()
        .sort_values(ascending=False)
        .head(top_k)
        .index.tolist()
    )

    raw = pd.DataFrame({"minute_of_day": np.arange(1440)})

    for ant in top_ant:
        s = (
            d[d["antenna_id"] == ant]
            .groupby("minute_of_day")["sig"]
            .median()
        )
        raw[f"sig_{ant}"] = raw["minute_of_day"].map(s).fillna(0)

    # ---- DISTANCE (per minute)
    dom = (
        d.groupby(["minute_of_day","antenna_id"])
         .size()
         .reset_index(name="n")
    )

    dom = dom.sort_values(["minute_of_day","n"], ascending=[True, False])
    dom = dom.drop_duplicates("minute_of_day")

    coords = (
        d.groupby(["minute_of_day","antenna_id"])
         .agg(lat=("recvDeployLat","median"),
              lon=("recvDeployLon","median"))
         .reset_index()
    )

    dom = dom.merge(coords, on=["minute_of_day","antenna_id"], how="left")

    dom["dist_m"] = haversine_m(
        base_lat, base_lon,
        dom["lat"], dom["lon"]
    )

    dist = pd.DataFrame({"minute_of_day": np.arange(1440)})
    dist = dist.merge(dom[["minute_of_day","dist_m"]], how="left")

# ---- detection
    dist["is_detected"] = dist["dist_m"].notna().astype(int)

# ---- visual flag
    dist["det_flag"] = np.where(dist["is_detected"] == 1, 1, -200)

# ---- fix for Label Studio (NO NaN allowed)
    dist["dist_m"] = dist["dist_m"].fillna(0)

    return merged, raw, dist

In [ ]:
tasks = []

pairs = (
    df[["bird_id","date"]]
    .drop_duplicates()
    .sort_values(["bird_id","date"])
)

for bird_id, day in pairs.itertuples(index=False):

    out = build_day(df, bird_id, day)
    if out is None:
        continue

    merged, raw, dist = out

    base = f"{bird_id}_{day}"

    merged_path = DATA_DIR / f"{base}_merged.csv"
    raw_path    = DATA_DIR / f"{base}_raw.csv"
    dist_path   = DATA_DIR / f"{base}_dist.csv"

    merged.to_csv(merged_path, index=False)
    raw.to_csv(raw_path, index=False)
    dist.to_csv(dist_path, index=False)

    # IMPORTANT:
    task = {
    "data": {
        "merged": f"/data/local-files/?d=test_2/data/{merged_path.name}",
        "raw":    f"/data/local-files/?d=test_2/data/{raw_path.name}",
        "dist":   f"/data/local-files/?d=test_2/data/{dist_path.name}",
        "bird_id": bird_id,
        "date": str(day)
    }
}

    tasks.append(task)

print("Tasks:", len(tasks))

In [9]:
json_path = OUT_ROOT / "tasks.jsonl"

with open(json_path, "w", encoding="utf-8") as f:
    for t in tasks:
        f.write(json.dumps(t) + "\n")

print("Saved:", json_path)

Saved: test_2\tasks.jsonl


In [8]:
#